# LangChain의 LCEL 방식 강의 추천 시스템 구성 구현
코드1 ) 단일 Agent
  - CourseAgent로 검색 + 답변까지 처리

코드2 ) 복수 Agent
  - PersonaAgent + QueryAgent + RetrieveAgent + ReplyAgent로 역할 분리후 처리

In [1]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-core
!pip install langchain-community
!pip install langchain-chroma
!pip install openai
!pip install python-dotenv
!pip install chromadb
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 688.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
import os
from dotenv import load_dotenv
from typing import List, Dict, Any
import shutil

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY가 없어요")

llm = ChatOpenAI(model="gpt-4.1-mini", temperature="0.5")

# Chroma용 Embedding 클래스
class STEmbedding:
  def __init__(self, model_name:str):
     self.model = SentenceTransformer(model_name)

  def embed_documents(self, texts:List[str]) -> List[List[float]]:
    return self.model.encode(texts).tolist()

  def embed_query(self, text:str) -> List[float]:
    return self.model.encode([text][0]).tolist()

embedding = STEmbedding("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# LangChain 강의 추천용 샘플 문서
docs = [
    Document(
        page_content="파이썬 기초 과정 / 프로그래밍 입문 / 변수,조건문,반복문,함수,리스트를 배운다.",
        metadata={"title": "파이썬 기초 과정", "level": "초급"}
    ),

    Document(
        page_content="데이터 분석 입문 / pandas,numpy,matplotlib를 사용해 데이터를 정리하고 시각화",
        metadata={"title": "데이터 분석 입문", "level": "초급~중급"}
    ),

    Document(
        page_content="머신러닝 실습 과정 / scikit-learn으로 회귀, 분류, 모델 평가를 실습한다.",
        metadata={"title": "머신러닝 실습 과정", "level": "중급"}
    ),

    Document(
        page_content="자연어 처리 과정 / 텍스트 전처리, 임베딩, 문서 검색, RAG 구조를 학습한다.",
        metadata={"title": "자연어 처리 과정", "level": "중급"}
    ),

    Document(
        page_content="MLOps 엔지니어 과정 / 모델 배포, Docker, FastAPI, CI/CD, 모니터링을 다룬다.",
        metadata={"title": "MLOps 엔지니어 과정", "level": "중급~고급"}
    ),

    Document(
        page_content="생성형 AI Agent 과정 / LangChain, Tool Calling, RAG, Multi-Agent 구조를 실습.",
        metadata={"title": "생성형 AI Agent 과정", "level": "중급~고급"}
    ),
]

In [5]:
CHROMA_DIR = './chroma_course'
shutil.rmtree(CHROMA_DIR, ignore_errors=True) # (실습용) 기존 폴더 삭제

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    collection_name="course_recomm",
    persist_directory=CHROMA_DIR
)

# 질문이 들어오면 관련 문서를 VectorDB에서 유사문서를 찾아주는 검색기 기능 수행
retriver = vectorstore.as_retriever(search_kwargs={"k": 3})

## AI 에이전트 (AI Agent)

### 코드1 ) 단일 Agent
  - CourseAgent로 검색 + 답변까지 처리

In [13]:
def course_agent(user_question:str) -> str:
  # RAG 검색
  found_docs = retriver.invoke(user_question) # 자동으로 embedding이 됨 - 편하니까 LangChain을 사용함.
  # print(found_docs)

  # 질문이 될수록 꾸며주기
  context = "\n".join(
      f"  - {doc.metadata['title']} ({doc.metadata['level']}) : {doc.page_content})" for doc in found_docs
  )
  # print(context)

  # LLM 답변 생성
  prompt = f"""
    너는 교육 과정 추천 도우미야
    아래 검색 결과를 참고해서 사용자에게 적절한 강의를 추천해 줘.

    사용자 질문 :
    {user_question}

    검색 결과 :
    {context}

    답변 조건 :
    한국어로 답변해라.
    추천 강의는 2 ~ 3개 정도.
    추천 이유를 설명해라.
  """
  response = llm.invoke(prompt)
  return response.content


# 실행하기
question = "파이썬은 조금 알고 있는데 AI쪽으로 공부하려면 어떤 강의가 좋을까?"
answer = course_agent(question)
print("질문 :",question)
print("답변 :",answer)

질문 : 파이썬은 조금 알고 있는데 AI쪽으로 공부하려면 어떤 강의가 좋을까?
답변 : 파이썬을 조금 알고 계시고 AI 쪽으로 공부를 원하신다면 다음 강의들을 추천드립니다.

1. **머신러닝 실습 과정 (중급)**  
   - 이유: 파이썬 기초를 어느 정도 알고 계시다면, 머신러닝의 기본 개념과 실습을 통해 AI의 핵심 기술을 익히기에 적합합니다. scikit-learn 라이브러리를 사용해 회귀, 분류, 모델 평가 등을 직접 실습하면서 실무 감각을 키울 수 있습니다.

2. **생성형 AI Agent 과정 (중급~고급)**  
   - 이유: AI 분야 중에서도 최근 주목받는 생성형 AI와 관련된 심화 과정입니다. LangChain, Tool Calling, RAG, Multi-Agent 구조 등을 다루어 최신 AI 기술 트렌드를 따라가고 싶다면 매우 유용합니다. 다만, 파이썬 기본기가 어느 정도 탄탄해야 하므로 머신러닝 실습 과정을 먼저 수강하는 것도 좋습니다.

이 두 과정을 순차적으로 학습하면 파이썬 기반 AI 실무 능력을 체계적으로 키울 수 있을 것입니다.


### 코드2 ) RAG + LangChain 기반 Agentic AI
  - 예) 사용자 질문 -> PersonaAgent -> QueryAgent -> RetriverAgent -> ReplyAgent -> 최총 추천 답변
  - 여러개의 에이전트를 사용하는것에 의의

In [15]:
# 사용자 질문에서 학습 수준, 관심 분야, 목표 등을 분석한다고 가정
def persona_agnet(user_question:str) -> str:
  prompt = f"""
    너는 persona_agnet야.
    아래 질문을 보고 학습자의 수준, 관심 분야, 목표를 한 문장으로 정리 해.

    사용자 질문 :
    {user_question}

    출력 :
  """
  return llm.invoke(prompt).content.strip()

#Persona Agent가 만든 학습자 정보를 바탕으로 Chroma 검색용 검색어 작성
def query_agnet(persona:str) -> str:
  prompt = f"""
    너는 query_agnet야.
    아래 학습자 정보를 보고 강의 검색에 사용할 검색어를 짧게 만들어.

    학습자 정보 :
    {persona}

    사용할 검색어 예 :
    머신러닝 실습, 자연어 처리 RAG

    출력 :
  """
  return llm.invoke(prompt).content.strip()

# Query Agent가 만든 검색어로 Chroma에서 관련 강의를 검색하는 에이전트
def retriver_agnet(query:str) -> str:
  found_docs = retriver.invoke(query)

  context = "\n".join(
      f"  - {doc.metadata['title']} ({doc.metadata['level']}) : {doc.page_content})"
      for doc in found_docs
  )
  return context

# 검색 결과를 종합해서 최종 답변 만드는 에이전트
def reply_agnet(user_question, persona, query, context:str) -> str:
  prompt = f"""
    너는 Reply Agent야.
    아래 정보를 보고 학습자에게 적장한 강의를 추천해 줘.

    사용자 정보 :
    {user_question}

    학습자 분석 :
    {persona}

    검색어 :
    {query}

    검색된 강의 :
    {context}

    답변 조건 :
    추천 강의는 2 ~ 3개를 한국어로 친절하게 안내해줘.
    왜 추천했는지 근거도 설명해라.
  """
  return llm.invoke(prompt).content

In [17]:
def agentic_ai_system(user_question:str) -> Dict[str, str]:
  # 전체 실행 함수 - 여러 Agent가 협력해서 최종 답변 생성
  persona = persona_agnet(user_question)
  query = query_agnet(persona)
  context = retriver_agnet(query)
  answer = reply_agnet(user_question, persona, query, context)

  return{
      'question': user_question,
      'persona': persona,
      'query': query,
      'context': context,
      'answer': answer
  }

# 실행
question = "데이터 분석을 조금 알고 있는데 다음은 무엇을 학습할까?"
result = agentic_ai_system(question)
print("질문 :", question)
print("1. PersonaAgnet 결과 :", result['persona'])
print("2. QueryAgnet 결과 :", result['query'])
print("3. RetriverAgnet 결과 :", result['context'])
print("4. ReplyAgnet 결과 :", result['answer'])

질문 : 데이터 분석을 조금 알고 있는데 다음은 무엇을 학습할까?
1. PersonaAgnet 결과 : 학습자는 데이터 분석에 기초 지식이 있으며, 이를 바탕으로 더 심화된 분석 기법이나 관련 도구를 학습하여 전문성을 높이고자 한다.
2. QueryAgnet 결과 : 심화 데이터 분석, 데이터 분석 도구
3. RetriverAgnet 결과 :   - 데이터 분석 입문 (초급~중급) : 데이터 분석 입문 / pandas,numpy,matplotlib를 사용해 데이터를 정리하고 시각화)
  - MLOps 엔지니어 과정 (중급~고급) : MLOps 엔지니어 과정 / 모델 배포, Docker, FastAPI, CI/CD, 모니터링을 다룬다.)
  - 자연어 처리 과정 (중급) : 자연어 처리 과정 / 텍스트 전처리, 임베딩, 문서 검색, RAG 구조를 학습한다.)
4. ReplyAgnet 결과 : 안녕하세요! 데이터 분석에 기초 지식이 있으시니, 다음 단계로 심화된 분석 기법과 관련 도구를 배우시면 전문성을 크게 높일 수 있을 것 같습니다.

추천 강의는 다음과 같습니다:

1. **자연어 처리 과정 (중급)**  
   - 이유: 데이터 분석의 심화 분야 중 하나인 자연어 처리는 텍스트 데이터를 다루는 데 필수적인 기술입니다. 텍스트 전처리, 임베딩, 문서 검색, RAG 구조 등 실무에 바로 적용 가능한 내용을 배울 수 있어, 데이터 분석 역량을 한층 강화할 수 있습니다.

2. **MLOps 엔지니어 과정 (중급~고급)**  
   - 이유: 분석한 모델을 실제 서비스에 안정적으로 배포하고 운영하는 과정인 MLOps는 데이터 분석 후 단계에서 매우 중요한 영역입니다. Docker, FastAPI, CI/CD, 모니터링 등 최신 도구와 기법을 익혀 데이터 분석 결과를 실무에 효과적으로 활용할 수 있습니다.

기초 과정은 이미 익히셨으니, 위 두 강의가 데이터 분석의 심화와 도구 활용 측면에서 큰 도움이 될 것입니다. 필요에 따라 두 강의를 순차적으로 수강하시면 데이